# 📚 arXiv Metadata Snapshot Exploration

This notebook explores the local parquet snapshot stored in `data/arxiv_metadata_snapshot/data/*.parquet` using **DuckDB** and **Pandas**.

### Sections:
1. **Schema & Total Count**
2. **Duplicate arXiv ID Check**
3. **Inspect Single Row (Detailed View)**
4. **Filter by arXiv Topic and Date Range**
5. **Search by Keyword with Topic & Date Filters**
6. **Aggregations & Summary Stats**
7. **Search by Specific arXiv ID**


In [13]:
import duckdb
import pandas as pd
import json

# Adjust pandas display limits for readability
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.max_columns', None)

# DATA_PATH = "../data/arxiv_metadata_snapshot/data/*.parquet"
DATA_PATH = "../data/arxiv_metadata_snapshot/data_clean/*.parquet"

con = duckdb.connect()
print(f"DuckDB version: {duckdb.__version__}")

DuckDB version: 1.5.5


## 1. Schema & Total Count

Check the schema of the parquet files and the total number of paper records.

In [3]:
# Inspect schema and column types
schema_df = con.sql(f"DESCRIBE SELECT * FROM '{DATA_PATH}'").df()
display(schema_df)

# Total count of papers
total_count = con.sql(f"SELECT count(*) AS total_papers FROM '{DATA_PATH}'").fetchone()[0]
print(f"\nTotal Papers in Dataset: {total_count:,}")

,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,submitter,VARCHAR,YES,None,None,None
2,authors,VARCHAR,YES,None,None,None
3,title,VARCHAR,YES,None,None,None
4,comments,VARCHAR,YES,None,None,None
5,journal-ref,VARCHAR,YES,None,None,None
6,doi,VARCHAR,YES,None,None,None
7,report-no,VARCHAR,YES,None,None,None
8,categories,VARCHAR,YES,None,None,None
9,license,VARCHAR,YES,None,None,None



Total Papers in Dataset: 3,164,528


## 2. Duplicate arXiv ID Check

Check if there are any duplicate records sharing the same arXiv `id`.


In [6]:
# Check for duplicate arXiv IDs
dup_check_df = con.sql(f"""
    SELECT
        count(*) AS total_records,
        count(DISTINCT id) AS distinct_ids,
        count(*) - count(DISTINCT id) AS duplicate_count
    FROM '{DATA_PATH}'
""").df()

display(dup_check_df)

dup_count = dup_check_df["duplicate_count"].iloc[0]
if dup_count > 0:
    print(f"\n⚠️ Found {dup_count:,} duplicate arXiv ID records.")
    sample_dups = con.sql(f"""
        SELECT
            id,
            count(*) AS occurrences
        FROM '{DATA_PATH}'
        GROUP BY id
        HAVING count(*) > 1
        ORDER BY occurrences DESC
        LIMIT 10
    """).df()
    print("Sample duplicate IDs:")
    display(sample_dups)
else:
    print("\n✅ No duplicate arXiv IDs found. All records are unique.")


,total_records,distinct_ids,duplicate_count
0,3164528,3164437,91



⚠️ Found 91 duplicate arXiv ID records.
Sample duplicate IDs:


,id,occurrences
0,math-ph/0512019,4
1,math-ph/0605055,4
2,math-ph/0509047,3
3,math-ph/0505039,3
4,math-ph/0203043,3
5,math-ph/0702049,3
6,math-ph/0303066,3
7,math-ph/0611044,3
8,math-ph/0606001,3
9,math-ph/9911045,2


In [9]:
# Filter parameters
DUPLICATED_ID = "math-ph/0605055"

query = f"""
    SELECT
        id,
        title,
        authors,
        categories,
        abstract,
        strftime(update_date, '%Y-%m-%d') AS update_date
    FROM '{DATA_PATH}'
    WHERE id LIKE '%{DUPLICATED_ID}%'
    ORDER BY update_date DESC
    LIMIT 10
"""

filtered_df = con.sql(query).df()
print(f"Duplicated entries for {DUPLICATED_ID}:")
display(filtered_df)

Duplicated entries for math-ph/0605055:


,id,title,authors,categories,abstract,update_date
0,math-ph/0605055,Galilean relativity and its invariant bilinear forms,Herintsitohaina Ratsimbarison,math-ph math.MP,We construct the family of bilinear forms gG on R3+1 for which Galilean\nboosts and spatial rotations are isometries. The key feature of these bilinear\nforms is that they are parametrized by a ...,2024-02-21
1,math-ph/0605055,Galilean relativity and its invariant bilinear forms,Herintsitohaina Ratsimbarison,math-ph math.MP,We construct the family of bilinear forms gG on R3+1 for which Galilean\nboosts and spatial rotations are isometries. The key feature of these bilinear\nforms is that they are parametrized by a ...,2024-02-21
2,math-ph/0605055,Galilean relativity and its invariant bilinear forms,Herintsitohaina Ratsimbarison,math-ph math.MP,We construct the family of bilinear forms gG on R3+1 for which Galilean\nboosts and spatial rotations are isometries. The key feature of these bilinear\nforms is that they are parametrized by a ...,2024-02-21
3,math-ph/0605055,Galilea relativity and its invariant bilinear forms,Herintsitohaina Ratsimbarison,math-ph math.MP,We construct the family of bilinear forms gG on R3+1 for which Galilea boosts\nand spatial rotations are isometries. The key feature of these bilinear forms\nis that they are parametrized by a G...,2007-05-23


## 3. Inspect Single Row (Detailed View)

Display all metadata fields for a single paper, including structured fields like `versions` and `authors_parsed`.


In [4]:
# Fetch a single row as a python dictionary / Series
sample_df = con.sql(f"""
    SELECT *
    FROM '{DATA_PATH}'
    WHERE id IS NOT NULL
    LIMIT 1
""").df()

# Display formatted key-value pairs for the single row
row = sample_df.iloc[0].to_dict()

print("=" * 80)
print(f"arXiv ID:     {row.get('id')}")
print(f"Title:        {row.get('title', '').strip()}")
print(f"Authors:      {row.get('authors')}")
print(f"Categories:   {row.get('categories')}")
print(f"Update Date:  {row.get('update_date')}")
print(f"License:      {row.get('license')}")
print(f"DOI:          {row.get('doi')}")
print(f"Journal Ref:  {row.get('journal-ref')}")
print(f"Comments:     {row.get('comments')}")
print("-" * 80)
print(f"Abstract:\n{row.get('abstract', '').strip()}")
print("-" * 80)
print(f"Versions:\n{row.get('versions')}")
print(f"Parsed Authors (First 3):\n{row.get('authors_parsed', [])[:3]}")
print("=" * 80)

arXiv ID:     0909.0774
Title:        Diophantine Geometry over Groups IX: Envelopes and Imaginaries
Authors:      Zlil Sela
Categories:   math.GR math.LO
Update Date:  2026-09-11 00:00:00
License:      http://arxiv.org/licenses/nonexclusive-distrib/1.0/
DOI:          None
Journal Ref:  None
Comments:     None
--------------------------------------------------------------------------------
Abstract:
This paper is the ninth in a sequence on the structure of sets of solutions to systems of equations in free and hyperbolic groups, projections of such sets (Diophantine sets), and the structure of definable sets in free and hyperbolic groups. In the ninth paper we associate a Diophantine set with a definable set, and view it as the Diophantine envelope of the definable set. We use the envelope and duo limit groups that were used in proving stability of the theory of free and torsion-free hyperbolic groups [Se9], to study definable equivalence relations, and in particular, to classify imagin

## 4. Filter by arXiv Topic and Date Range

Filter papers by:
- **Topic / Category**: e.g. Computational Linguistics (`cs.CL`), Artificial Intelligence (`cs.AI`), Machine Learning (`stat.ML` / `cs.LG`)
- **Date Range**: `update_date >= '2024-01-01'` (or specific year/month)


In [5]:
# Filter parameters
TARGET_TOPIC = "cs.CL"
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

query = f"""
    SELECT
        id,
        title,
        authors,
        categories,
        strftime(update_date, '%Y-%m-%d') AS update_date
    FROM '{DATA_PATH}'
    WHERE categories LIKE '%{TARGET_TOPIC}%'
      AND update_date >= '{START_DATE}'
      AND update_date <= '{END_DATE}'
    ORDER BY update_date DESC
    LIMIT 10
"""

filtered_df = con.sql(query).df()
print(f"Results for category containing '{TARGET_TOPIC}' between {START_DATE} and {END_DATE}:")
display(filtered_df)

Results for category containing 'cs.CL' between 2024-01-01 and 2024-12-31:


,id,title,authors,categories,update_date
0,2207.08112,United States Politicians' Tone Became More Negative with 2016 Primary\n Campaigns,"Jonathan K\""ulz, Andreas Spitz, Ahmad Abu-Akel, Stephan G\""unnemann,\n Robert West",cs.CL,2024-12-31
1,2304.09542,Is ChatGPT Good at Search? Investigating Large Language Models as\n Re-Ranking Agents,"Weiwei Sun, Lingyong Yan, Xinyu Ma, Shuaiqiang Wang, Pengjie Ren,\n Zhumin Chen, Dawei Yin, Zhaochun Ren",cs.CL cs.IR,2024-12-31
2,2305.10383,Large-Scale Text Analysis Using Generative Language Models: A Case Study\n in Discovering Public Value Expressions in AI Patents,"Sergio Pelaez, Gaurav Verma, Barbara Ribeiro, Philip Shapira",cs.CL cs.IR,2024-12-31
3,2307.10246,Deep Neural Networks and Brain Alignment: Brain Encoding and Decoding\n (Survey),"Subba Reddy Oota, Zijiao Chen, Manish Gupta, Raju S. Bapi, Gael\n Jobard, Frederic Alexandre, Xavier Hinaut",q-bio.NC cs.AI cs.CL cs.CV cs.HC cs.LG,2024-12-31
4,2310.05650,ReZG: Retrieval-Augmented Zero-Shot Counter Narrative Generation for\n Hate Speech,"Shuyu Jiang, Wenyi Tang, Xingshu Chen, Rui Tang, Haizhou Wang and\n Wenxian Wang",cs.CL,2024-12-31
5,2310.09881,In-Context Learning with Iterative Demonstration Selection,"Chengwei Qin, Aston Zhang, Chen Chen, Anirudh Dagar, Wenming Ye",cs.CL cs.AI,2024-12-31
6,2311.05844,Face-StyleSpeech: Enhancing Zero-shot Speech Synthesis from Face Images\n with Improved Face-to-Speech Mapping,"Minki Kang, Wooseok Han, Eunho Yang",cs.CV cs.AI cs.CL cs.MM cs.SD eess.AS,2024-12-31
7,2311.09367,A Survey on Online User Aggression: Content Detection and Behavioral\n Analysis on Social Media,"Swapnil Mane, Suman Kundu, Rajesh Sharma",cs.CL,2024-12-31
8,2312.07395,A Simple Recipe for Contrastively Pre-training Video-First Encoders\n Beyond 16 Frames,"Pinelopi Papalampidi, Skanda Koppula, Shreya Pathak, Justin Chiu, Joe\n Heyward, Viorica Patraucean, Jiajun Shen, Antoine Miech, Andrew Zisserman,\n Aida Nematzadeh",cs.CV cs.CL,2024-12-31
9,2402.10835,Time Series Forecasting with LLMs: Understanding and Enhancing Model\n Capabilities,"Hua Tang, Chong Zhang, Mingyu Jin, Qinkai Yu, Zhenting Wang, Xiaobo\n Jin, Yongfeng Zhang, Mengnan Du",cs.CL,2024-12-31


## 5. Search by Keyword (Title / Abstract) with Topic & Date Filters

Perform case-insensitive keyword search on titles and abstracts for specific topics within a date window.


In [6]:
KEYWORD = "quantum"
CATEGORY = "quant-ph"
START_DATE = "2023-01-01"

query_keyword = f"""
    SELECT
        id,
        title,
        categories,
        strftime(update_date, '%Y-%m-%d') AS update_date,
        SUBSTR(abstract, 1, 150) || '...' AS abstract_preview
    FROM '{DATA_PATH}'
    WHERE categories LIKE '%{CATEGORY}%'
      AND update_date >= '{START_DATE}'
      AND (lower(title) LIKE '%{KEYWORD.lower()}%' OR lower(abstract) LIKE '%{KEYWORD.lower()}%')
    ORDER BY update_date DESC
    LIMIT 5
"""

keyword_df = con.sql(query_keyword).df()
display(keyword_df)

,id,title,categories,update_date,abstract_preview
0,2201.01562,Efficiently simulating the work distribution of multiple identical bosons with boson sampling,quant-ph,2026-09-11,"Boson sampling has been theoretically proposed and experimentally demonstrated to show quantum computational advantages. However, it still lacks the d..."
1,2302.00821,Designing a Hybrid Digital / Analog Quantum Physics Emulator as Open Hardware,quant-ph,2026-09-11,Existing approaches to emulating quantum computing algorithms using classical electronic hardware are limited by exponential scaling limitations in sp...
2,2403.05491,Slow Light Augmented Unbalanced Interferometry for Extreme Enhancement in Sensitivity of Measuring Frequency Shift in a Laser,quant-ph,2026-09-11,We demonstrate a slow-light augmented unbalanced Mach-Zehnder interferometer (MZI) which can be used to enhance very significantly the sensitivity of ...
3,2406.11088,"Sixth-order time-convolutionless master equation and beyond: Late-time resummations, two types of divergences, and the limits of validity",quant-ph,2026-09-11,Perturbative master equations are essential for modeling open quantum systems but often exhibit late-time divergences when environmental correlations ...
4,2412.02982,Quantum birthmarks: Ergodicity breaking beyond scarring,quant-ph,2026-09-11,A hallmark of classical ergodicity is the complete loss of memory of the initial conditions due to eventual uniform covering of {\it a priori} availab...


## 6. Aggregations & Summary Stats

Analyze the top categories and distribution of updates over time.


In [7]:
# Top 10 most common arXiv category combinations
top_categories_df = con.sql(f"""
    SELECT
        categories,
        COUNT(*) AS count
    FROM '{DATA_PATH}'
    GROUP BY categories
    ORDER BY count DESC
    LIMIT 10
""").df()

print("Top 10 Category Combinations:")
display(top_categories_df)

# Papers updated per year
yearly_counts_df = con.sql(f"""
    SELECT
        EXTRACT(YEAR FROM update_date) AS year,
        COUNT(*) AS paper_count
    FROM '{DATA_PATH}'
    WHERE update_date IS NOT NULL
    GROUP BY year
    ORDER BY year DESC
    LIMIT 10
""").df()

print("\nPapers by Update Year:")
display(yearly_counts_df)

Top 10 Category Combinations:


,categories,count
0,cs.CV,96297
1,astro-ph,86896
2,hep-ph,86612
3,quant-ph,84333
4,hep-th,62770
5,cond-mat.mtrl-sci,46878
6,math.AP,42630
7,cs.CL,41760
8,cond-mat.mes-hall,38599
9,astro-ph.GA,38139



Papers by Update Year:


,year,paper_count
0,2026,357129
1,2025,302854
2,2024,245191
3,2023,209741
4,2022,185525
5,2021,183152
6,2020,171859
7,2019,150361
8,2018,130514
9,2017,117918


## 7. Search by Specific arXiv ID

Lookup a paper directly by its arXiv ID (e.g. `1706.03762` for *Attention Is All You Need*, or older formats like `hep-th/9711200`).


In [8]:
# Specify arXiv ID to search
TARGET_ARXIV_ID = "2609.05412"

paper_df = con.sql(f"""
    SELECT *
    FROM '{DATA_PATH}'
    WHERE id = '{TARGET_ARXIV_ID}'
""").df()

if len(paper_df) == 0:
    print(f"No paper found for arXiv ID: {TARGET_ARXIV_ID}")
else:
    paper = paper_df.iloc[0].to_dict()
    print("=" * 80)
    print(f"ID:          {paper.get('id')}")
    print(f"Title:       {paper.get('title', '').strip()}")
    print(f"Authors:     {paper.get('authors')}")
    print(f"Categories:  {paper.get('categories')}")
    print(f"Update Date: {paper.get('update_date')}")
    print(f"DOI:         {paper.get('doi')}")
    print(f"Comments:    {paper.get('comments')}")
    print(f"Links:       https://arxiv.org/abs/{paper.get('id')} | https://arxiv.org/pdf/{paper.get('id')}")
    print("-" * 80)
    print(f"Abstract:\n{paper.get('abstract', '').strip()}")
    print("=" * 80)

ID:          2609.05412
Title:       Tracing the Cosmic Origins: Machine Learning Reconstruction of the Primordial Density Field from EoR Observations
Authors:     Anchal Saxena, P. Daniel Meerburg, Guochao Sun, Tzu-Ching Chang and Llu\'is Mas-Ribas
Categories:  astro-ph.CO astro-ph.IM
Update Date: 2026-09-07 00:00:00
DOI:         None
Comments:    18 pages, 13 figures. Submitted to the Monthly Notices of the Royal Astronomical Society (MNRAS)
Links:       https://arxiv.org/abs/2609.05412 | https://arxiv.org/pdf/2609.05412
--------------------------------------------------------------------------------
Abstract:
Reconstructing the initial conditions of the Universe from late-time tracers would unlock cosmological information buried by non-linear structure formation and astrophysics. We reconstruct the initial density field at $z\sim300$ from simulated 21-cm and CO(1-0) line-intensity maps at $z\sim8$ generated with LIMFAST. Using a three-dimensional U-Net, we reconstruct the initial co